In [2]:
import pandas as pd
from google.colab import files

print("Sube tus archivos CSV:")
uploaded = files.upload()

dfs = {}

for filename in uploaded.keys():
    print(f"Archivo subido: {filename}")
    dfs[filename] = pd.read_csv(filename)

dfs.keys()

master = dfs["master.csv"]


Sube tus archivos CSV:


Saving master.csv to master.csv
Archivo subido: master.csv


,count
satisfaccion_cliente,
Neutral,9593
Baja satisfacción,4736
Alta satisfacción,696


In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1. Importancias obtenidas del modelo
# ============================================================

importancias = {
    "monto_maximo": 0.084690,
    "recibe_remesas": 0.078560,
    "utilizacion_promedio": 0.064548,
    "n_transacciones": 0.056276,
    "monto_total": 0.054234,
    "cashback_total": 0.053843,
    "ticket_promedio": 0.050921,
    "dias_desde_ultimo_login": 0.050062,
    "usa_hey_shop": 0.044159,
    "pct_mensajes_positivos": 0.043005,
    "pct_patron_atipico_tx": 0.040301,
    "patron_uso_atipico": 0.038111,
    "nomina_domiciliada": 0.037680,
    "total_positivos": 0.036864,
    "n_productos_cancelados": 0.025953
}

# ============================================================
# 2. Dirección esperada del efecto sobre abandono
# ============================================================
# +1: aumenta el riesgo de abandono
# -1: reduce el riesgo de abandono

signos = {
    "monto_maximo": -1,
    "recibe_remesas": -1,
    "utilizacion_promedio": 1,
    "n_transacciones": -1,
    "monto_total": -1,
    "cashback_total": -1,
    "ticket_promedio": -1,
    "dias_desde_ultimo_login": 1,
    "usa_hey_shop": -1,
    "pct_mensajes_positivos": -1,
    "pct_patron_atipico_tx": 1,
    "patron_uso_atipico": 1,
    "nomina_domiciliada": -1,
    "total_positivos": -1,
    "n_productos_cancelados": 1
}

# ============================================================
# 3. Proporción de baja satisfacción por cluster
# ============================================================

baja_satisfaccion_cluster = {
    -1: 0.3362,
     0: 0.2927,
     1: 0.3263,
     2: 0.3255,
     3: 0.3201,
     4: 0.3518,
     5: 0.3281,
     6: 0.2986,
     7: 0.3504,
     8: 0.3745,
     9: 0.2804,
    10: 0.1918,
    11: 0.3358,
    12: 0.2701,
    13: 0.3043,
    14: 0.3652,
    15: 0.1760,
    16: 0.2802,
    17: 0.0000,
    18: 0.8369
}

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def preparar_modelo_abandono(
    df_base,
    importancias,
    signos,
    baja_satisfaccion_cluster,
    beta_0=-1.5,
    lambda_cluster=0.7,
    eta=1.2,
    epsilon=0.001
):
    """
    Prepara los elementos necesarios para evaluar clientes manuales.
    Ajusta el escalador usando la base completa.
    """

    variables = list(importancias.keys())

    variables_disponibles = [
        v for v in variables
        if v in df_base.columns
    ]

    if len(variables_disponibles) == 0:
        raise ValueError("Ninguna variable de importancia está en df_base.")

    X_base = df_base[variables_disponibles].copy()

    for col in X_base.select_dtypes(include=["bool"]).columns:
        X_base[col] = X_base[col].astype(int)

    X_base = X_base.fillna(0)

    scaler = StandardScaler()
    scaler.fit(X_base)

    suma_importancias = sum(importancias[v] for v in variables_disponibles)

    pesos = {
        v: signos[v] * (importancias[v] / suma_importancias)
        for v in variables_disponibles
    }

    modelo = {
        "variables": variables_disponibles,
        "scaler": scaler,
        "pesos": pesos,
        "baja_satisfaccion_cluster": baja_satisfaccion_cluster,
        "beta_0": beta_0,
        "lambda_cluster": lambda_cluster,
        "eta": eta,
        "epsilon": epsilon
    }

    return modelo

In [4]:
def predecir_cliente_manual(
    cliente,
    modelo
):
    """
    Predice la probabilidad de abandono para un cliente ingresado manualmente.

    cliente debe ser un diccionario con:
    - las variables numéricas
    - cluster
    - satisfaccion_cliente
    """

    variables = modelo["variables"]
    scaler = modelo["scaler"]
    pesos = modelo["pesos"]
    baja_satisfaccion_cluster = modelo["baja_satisfaccion_cluster"]
    beta_0 = modelo["beta_0"]
    lambda_cluster = modelo["lambda_cluster"]
    eta = modelo["eta"]
    epsilon = modelo["epsilon"]

    # Convertir cliente a DataFrame
    df_cliente = pd.DataFrame([cliente])

    # Verificar variables faltantes
    for v in variables:
        if v not in df_cliente.columns:
            df_cliente[v] = 0

    X_cliente = df_cliente[variables].copy()

    for col in X_cliente.select_dtypes(include=["bool"]).columns:
        X_cliente[col] = X_cliente[col].astype(int)

    X_cliente = X_cliente.fillna(0)

    # Estandarizar usando el scaler ajustado con la base completa
    X_z = pd.DataFrame(
        scaler.transform(X_cliente),
        columns=variables
    )

    # Score por variables
    score_variables = 0

    for v in variables:
        score_variables += pesos[v] * X_z.loc[0, v]

    # Score por cluster
    cluster = cliente.get("cluster", -1)

    r_c = baja_satisfaccion_cluster.get(
        cluster,
        np.mean(list(baja_satisfaccion_cluster.values()))
    )

    score_cluster = np.log(
        (r_c + epsilon) /
        (1 - r_c + epsilon)
    )

    # Score por satisfacción individual
    satisfaccion = cliente.get("satisfaccion_cliente", "Neutral")

    if satisfaccion == "Alta satisfacción":
        score_satisfaccion = -eta
    elif satisfaccion == "Baja satisfacción":
        score_satisfaccion = eta
    else:
        score_satisfaccion = 0

    # Puntaje total
    Z = (
        beta_0
        + score_variables
        + lambda_cluster * score_cluster
        + score_satisfaccion
    )

    # Probabilidad
    prob_abandono = 1 / (1 + np.exp(-Z))

    # Clasificación
    if prob_abandono < 0.25:
        nivel = "Bajo"
    elif prob_abandono < 0.50:
        nivel = "Medio"
    elif prob_abandono < 0.75:
        nivel = "Alto"
    else:
        nivel = "Crítico"

    resultado = {
        "cluster": cluster,
        "satisfaccion_cliente": satisfaccion,
        "score_variables": score_variables,
        "score_cluster": score_cluster,
        "score_satisfaccion": score_satisfaccion,
        "Z_abandono": Z,
        "prob_abandono": prob_abandono,
        "nivel_riesgo_abandono": nivel
    }

    return resultado

In [8]:
df = master

modelo_abandono = preparar_modelo_abandono(
    df_base=df,
    importancias=importancias,
    signos=signos,
    baja_satisfaccion_cluster=baja_satisfaccion_cluster,
    beta_0=-1.5,
    lambda_cluster=0.7,
    eta=1.2,
    epsilon=0.001
)

In [11]:
cliente_manual = {
    "monto_maximo": 2500,
    "recibe_remesas": 0,
    "utilizacion_promedio": 0.75,
    "n_transacciones": 4,
    "monto_total": 4200,
    "cashback_total": 15,
    "ticket_promedio": 1050,
    "dias_desde_ultimo_login": 45,
    "usa_hey_shop": 0,
    "pct_mensajes_positivos": 0.10,
    "pct_patron_atipico_tx": 0.35,
    "patron_uso_atipico": 1,
    "nomina_domiciliada": 0,
    "total_positivos": 2,
    "n_productos_cancelados": 1,
    "cluster": 18,
    "satisfaccion_cliente": "Baja satisfacción"
}

resultado = predecir_cliente_manual(
    cliente=cliente_manual,
    modelo=modelo_abandono
)

print(f"Probabilidad de abandono: {resultado['prob_abandono']:.2f}")
print(f"Satisfacción del cliente: {resultado['satisfaccion_cliente']}")

Probabilidad de abandono: 0.86
Satisfacción del cliente: Baja satisfacción
